# Section 1. LLM API 호출의 기본 구조

이 섹션에서는 LLM을 함수처럼 호출하는 최소 단위를 배웁니다. 중요한 것은 모델 이름, 입력 prompt, 출력 text, 비용이 발생하는 호출 지점을 구분하는 것입니다.

실습 목표:

- 같은 입력이라도 prompt 지시가 바뀌면 출력이 달라지는 것을 확인합니다.
- API 호출 결과를 사람이 읽는 text로 먼저 받아봅니다.
- 이후 섹션에서 구조화 출력과 RAG로 확장할 준비를 합니다.


## API key 입력

각 실습 노트북은 독립적으로 실행됩니다. 아래 셀의 따옴표 안에 수업용 OpenAI API key를 붙여넣고 실행하세요.

- key가 들어간 노트북은 그대로 공유하지 않습니다.
- 제출하거나 화면 공유하기 전에는 key 문자열을 지웁니다.
- 이 자료에서는 `.env` 파일을 쓰지 않습니다. 학생이 열어야 할 파일 수를 줄이기 위해 노트북 안에서 직접 입력합니다.


In [ ]:
import os

OPENAI_API_KEY = "여기에_수업용_API_KEY를_붙여넣으세요"
OPENAI_MODEL = "gpt-5.4-mini"

if not OPENAI_API_KEY or OPENAI_API_KEY.startswith("여기에_"):
    raise ValueError("OPENAI_API_KEY 자리에 수업용 API key를 붙여넣은 뒤 다시 실행하세요.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key가 현재 노트북 실행 세션에 설정되었습니다.")


## 실습 데이터

아래 메모들은 이후 섹션에서도 반복해서 쓰는 작은 업무 문서입니다. 지금은 LLM에게 요약과 분류를 시켜보는 입력으로 사용합니다.


In [ ]:
NOTES = [
    {
        "note_id": "support_triage",
        "text": "Support triage routes billing requests to the finance queue. Urgent security issues go to the incident channel. Uncertain cases use a fallback path for review.",
    },
    {
        "note_id": "api_onboarding",
        "text": "API onboarding requires authentication, environment variables, awareness of rate limits, and one minimal smoke test before larger experiments.",
    },
    {
        "note_id": "rag_quality",
        "text": "A grounded RAG answer should cite the retrieved source, quote only text that appears in that source, and say not answered when the corpus lacks evidence.",
    },
]


## 첫 호출: 메모 하나 요약하기

prompt는 모델에게 맡길 일을 설명하는 요청문입니다. 아래 셀은 API onboarding 메모를 두 문장 이하로 요약합니다.


In [ ]:
from openai import OpenAI

client = OpenAI()
note = NOTES[1]
prompt = f"""
다음 메모를 한국어로 두 문장 이하로 요약하세요.

메모 ID: {note['note_id']}
내용: {note['text']}
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=prompt,
    max_output_tokens=250,
)
print(response.output_text)


## 두 번째 호출: 역할과 형식 지시 추가

LLM은 자동으로 정답 형식을 보장하지 않습니다. 그래도 prompt에 역할, 출력 항목, 금지사항을 명확히 쓰면 결과가 훨씬 안정됩니다.


In [ ]:
prompt = f"""
당신은 인턴 학습 자료를 준비하는 조교입니다.
아래 메모를 보고 학생에게 알려줄 체크리스트를 한국어 bullet 3개로 작성하세요.
문서에 없는 내용은 추가하지 마세요.

내용: {note['text']}
"""

response = client.responses.create(
    model=OPENAI_MODEL,
    input=prompt,
    max_output_tokens=300,
)
print(response.output_text)


## 결과 확인

실행 후 아래를 스스로 확인합니다.

- API key가 들어간 셀을 실행해야만 호출이 진행됩니다.
- `max_output_tokens`는 출력 길이를 제한합니다.
- prompt에 없는 사실을 모델이 보태는지 주의해서 봅니다.
- 다음 섹션에서는 text가 아니라 정해진 schema에 맞는 JSON을 받습니다.
